# 🏦 IndoSynth Gramin Bank — EDA Preprocessing

This notebook covers all **data-cleaning and preprocessing steps** before the visual EDA.

### Pipeline Overview
| Step | Description |
|------|-------------|
| 1 | Data Loading & Shape Overview |
| 2 | Data Types & Info |
| 3 | Missing Value Analysis |
| 4 | Duplicate Detection |
| 5 | Date Column Conversion |
| 6 | Categorical Standardization |
| 7 | Numerical Outlier Detection (IQR) |
| 8 | Feature Engineering (Derived Columns) |
| 9 | Export Cleaned Data |

---
## 📦 Imports & Configuration

In [1]:
# ─────────────────────────────────────────────────────────────
# IMPORTS & CONFIGURATION
# ─────────────────────────────────────────────────────────────
# Import the core libraries required for the preprocessing:
#   • pandas  — for data manipulation and CSV reading
#   • numpy   — for numerical operations and NaN handling
#   • warnings — to suppress unnecessary warning messages
#   • os      — for file/directory path operations
#
# We also define two key paths:
#   DATA_PATH  → root folder containing all raw CSV files
#   CLEAN_PATH → output folder where cleaned CSVs will be saved
# ─────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import warnings
import os

warnings.filterwarnings('ignore')

# ── Paths ──
DATA_PATH = r"B:\Major Project\data"
CLEAN_PATH = os.path.join(r"B:\Major Project", "eda", "cleaned_data")
os.makedirs(CLEAN_PATH, exist_ok=True)

print("✅ Imports loaded & output directory ready.")

✅ Imports loaded & output directory ready.


---
## 📥 Step 1: Data Loading & Shape Overview

In [2]:
# ─────────────────────────────────────────────────────────────
# STEP 1a: LOAD ALL 9 DATASETS FROM CSV FILES
# ─────────────────────────────────────────────────────────────
# Read each of the 9 CSV files (regions, loan_types, branches,
# employees, customers, credit_history, loan_applications,
# loan_payments, transactions) into separate DataFrames.
#
# We also create an `all_tables` dictionary mapping table names
# to their DataFrames — this allows us to iterate over all
# tables easily in subsequent preprocessing steps.
# ─────────────────────────────────────────────────────────────

regions        = pd.read_csv(f"{DATA_PATH}/regions.csv")
loan_types     = pd.read_csv(f"{DATA_PATH}/loan_types.csv")
branches       = pd.read_csv(f"{DATA_PATH}/branches.csv")
employees      = pd.read_csv(f"{DATA_PATH}/employees.csv")
customers      = pd.read_csv(f"{DATA_PATH}/customers.csv")
credit_history = pd.read_csv(f"{DATA_PATH}/credit_history.csv")
loan_apps      = pd.read_csv(f"{DATA_PATH}/loan_applications.csv")
loan_payments  = pd.read_csv(f"{DATA_PATH}/loan_payments.csv")
transactions   = pd.read_csv(f"{DATA_PATH}/transactions.csv")

# Dictionary for easy iteration
all_tables = {
    'regions':        regions,
    'loan_types':     loan_types,
    'branches':       branches,
    'employees':      employees,
    'customers':      customers,
    'credit_history': credit_history,
    'loan_apps':      loan_apps,
    'loan_payments':  loan_payments,
    'transactions':   transactions,
}

print("✅ All 9 datasets loaded successfully!")

✅ All 9 datasets loaded successfully!


In [3]:
# ─────────────────────────────────────────────────────────────
# STEP 1b: SHAPE & MEMORY OVERVIEW
# ─────────────────────────────────────────────────────────────
# Display the number of rows, columns, and memory usage for
# each table in a clean tabular format.
#
# This helps us understand:
#   • The scale of each dataset (e.g., transactions = 1.5M rows)
#   • Total memory footprint across all tables
#   • Which tables are the largest and may need careful handling
# ─────────────────────────────────────────────────────────────

summary_rows = []
for name, df in all_tables.items():
    mem = df.memory_usage(deep=True).sum() / (1024 ** 2)
    summary_rows.append({'Table': name, 'Rows': f"{df.shape[0]:,}",
                         'Columns': df.shape[1], 'Memory (MB)': f"{mem:.2f}"})

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

total_rows = sum(df.shape[0] for df in all_tables.values())
print(f"\n📊 Total records across all tables: {total_rows:,}")

,Table,Rows,Columns,Memory (MB)
0,regions,40,6,0.00
1,loan_types,10,11,0.00
2,branches,250,11,0.05
3,employees,"2,000",16,0.44
4,customers,"100,000",29,41.21
5,credit_history,"100,000",12,10.64
6,loan_apps,"300,000",20,64.35
7,loan_payments,"900,000",13,109.45
8,transactions,"1,500,000",13,268.46



📊 Total records across all tables: 2,902,300


---
## 📋 Step 2: Data Types & Column Summary

In [4]:
# ─────────────────────────────────────────────────────────────
# STEP 2: DATA TYPES & COLUMN SUMMARY
# ─────────────────────────────────────────────────────────────
# For each table, display:
#   • Column name, data type, non-null count, and null count
#   • First 3 rows as a preview
#
# This step is crucial because:
#   • Date columns are initially loaded as 'object' (string)
#     and need conversion to datetime
#   • Numeric columns may contain unexpected types
#   • We can spot columns that need cleaning at a glance
# ─────────────────────────────────────────────────────────────

for name, df in all_tables.items():
    print(f"\n{'═' * 55}")
    print(f"📌 {name.upper()} — {df.shape[0]:,} rows × {df.shape[1]} cols")
    print(f"{'═' * 55}")
    
    info_rows = []
    for col in df.columns:
        info_rows.append({
            'Column': col,
            'Dtype': str(df[col].dtype),
            'Non-Null': f"{df[col].notna().sum():,}",
            'Nulls': f"{df[col].isna().sum():,}"
        })
    display(pd.DataFrame(info_rows))
    print(f"\n  First 3 rows:")
    display(df.head(3))


═══════════════════════════════════════════════════════
📌 REGIONS — 40 rows × 6 cols
═══════════════════════════════════════════════════════


,Column,Dtype,Non-Null,Nulls
0,region_id,int64,40,0
1,region_name,str,40,0
2,zone,str,40,0
3,primary_state,str,40,0
4,states_covered,str,40,0
5,num_states,int64,40,0



  First 3 rows:


,region_id,region_name,zone,primary_state,states_covered,num_states
0,1,Lucknow Central Region,Central,Uttar Pradesh,Uttar Pradesh,1
1,2,Bhopal Central Region,Central,Madhya Pradesh,Madhya Pradesh,1
2,3,Raipur Central Region,Central,Chhattisgarh,Chhattisgarh,1



═══════════════════════════════════════════════════════
📌 LOAN_TYPES — 10 rows × 11 cols
═══════════════════════════════════════════════════════


...

,payment_id,application_id,customer_id,payment_number,due_date,payment_date,emi_amount,principal_paid,interest_paid,penalty_amount,payment_status,days_late,outstanding_balance
0,1,1,12114,1,2021-11-26,2021-11-26,71111.09,51081.99,20029.10,0.00,Paid,0,1936918.01
1,2,1,12114,2,2021-12-26,2022-01-02,71111.09,51596.64,19514.45,331.85,Late,7,1885321.37
2,3,1,12114,3,2022-01-25,2022-01-27,71111.09,52116.48,18994.61,94.81,Paid,2,1833204.89



═══════════════════════════════════════════════════════
📌 TRANSACTIONS — 1,500,000 rows × 13 cols
═══════════════════════════════════════════════════════


,Column,Dtype,Non-Null,Nulls
0,transaction_id,int64,"1,500,000",0
1,customer_id,int64,"1,500,000",0
2,transaction_date,str,"1,500,000",0
3,transaction_type,str,"1,500,000",0
4,transaction_mode,str,"1,500,000",0
5,amount,float64,"1,500,000",0
6,balance_after,float64,"1,500,000",0
7,category,str,"1,500,000",0
8,merchant_name,str,"1,016,031","483,969"
9,reference_number,str,"1,500,000",0



  First 3 rows:


,transaction_id,customer_id,transaction_date,transaction_type,transaction_mode,amount,balance_after,category,merchant_name,reference_number,description,branch_id,status
0,1,1,2018-02-09,Debit,ATM,4037.38,38417.87,Fuel,Indian Oil,NEFT804734457381,Debit - Fuel via ATM,213.0,Success
1,2,1,2018-09-04,Credit,UPI,26763.56,65181.43,Business Income,NaN,IMPS143558679682,Credit - Business Income via UPI,NaN,Success
2,3,1,2018-10-30,Credit,UPI,15399.85,80581.28,Salary,NaN,NEFT765949342090,Credit - Salary via UPI,NaN,Success


---
## 🔍 Step 3: Missing Value Analysis

In [5]:
# ─────────────────────────────────────────────────────────────
# STEP 3a: DETECT MISSING VALUES ACROSS ALL TABLES
# ─────────────────────────────────────────────────────────────
# Scan every column in every table for NULL/NaN values.
# For columns with missing data, display:
#   • The column name
#   • Count of missing values
#   • Percentage of total rows affected
#
# This helps us decide which columns need imputation,
# filling, or can be left as-is (e.g., rejection_reason
# is NULL for non-rejected loans — that's expected).
# ─────────────────────────────────────────────────────────────

any_missing = False
for name, df in all_tables.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        any_missing = True
        pct = (missing / len(df) * 100).round(2)
        print(f"\n📌 {name.upper()}")
        miss_df = pd.DataFrame({'Column': missing.index,
                                'Missing': missing.values,
                                '% of Total': pct.values})
        display(miss_df)

if not any_missing:
    print("✅ No missing values found in any table!")


📌 LOAN_TYPES


,Column,Missing,% of Total
0,collateral_type,6,60.0



📌 LOAN_APPS


,Column,Missing,% of Total
0,loan_amount_approved,156701,52.23
1,processing_fee,156701,52.23
2,emi_amount,156701,52.23
3,rejection_reason,158224,52.74
4,collateral_type,180134,60.04
5,disbursement_date,168162,56.05



📌 LOAN_PAYMENTS


,Column,Missing,% of Total
0,payment_date,68299,7.59



📌 TRANSACTIONS


,Column,Missing,% of Total
0,merchant_name,483969,32.26
1,branch_id,1125416,75.03


### 🔧 Handle Missing Values

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 3b: HANDLE MISSING VALUES
# ─────────────────────────────────────────────────────────────
# Apply context-aware imputation for each column with NULLs:
#
# loan_applications:
#   • rejection_reason   → fill with 'N/A' (NULL = not rejected)
#   • disbursement_date  → keep NULL (NULL = not yet disbursed)
#   • loan_amount_approved, emi_amount, interest_rate_pct,
#     processing_fee     → fill with 0 (NULL = rejected/pending)
#   • collateral_type    → fill with 'None' (unsecured loans)
#
# transactions:
#   • branch_id      → fill with -1 (online/digital transactions)
#   • merchant_name  → fill with 'Unknown' (non-merchant txns)
#
# Strategy: We use domain knowledge to fill values rather than
# dropping rows, since these NULLs are structurally expected.
# ─────────────────────────────────────────────────────────────

# loan_apps: rejection_reason is NULL when status != 'Rejected' → expected
if 'rejection_reason' in loan_apps.columns:
    null_rej = loan_apps['rejection_reason'].isna().sum()
    non_rejected = (loan_apps['status'] != 'Rejected').sum()
    print(f"• loan_apps.rejection_reason: {null_rej:,} NULLs "
          f"({non_rejected:,} non-rejected) → Expected, filling with 'N/A'")
    loan_apps['rejection_reason'] = loan_apps['rejection_reason'].fillna('N/A')

# loan_apps: disbursement_date is NULL when not disbursed → expected
if 'disbursement_date' in loan_apps.columns:
    null_disb = loan_apps['disbursement_date'].isna().sum()
    print(f"• loan_apps.disbursement_date: {null_disb:,} NULLs → "
          f"Expected for non-disbursed loans, keeping as-is")

# loan_apps: loan_amount_approved can be NULL for rejected
if 'loan_amount_approved' in loan_apps.columns:
    null_amt = loan_apps['loan_amount_approved'].isna().sum()
    print(f"• loan_apps.loan_amount_approved: {null_amt:,} NULLs → "
          f"Filling with 0 for non-approved")
    loan_apps['loan_amount_approved'] = loan_apps['loan_amount_approved'].fillna(0)

# loan_apps: emi_amount, interest_rate_pct, processing_fee can be NULL for rejected
for col in ['emi_amount', 'interest_rate_pct', 'processing_fee']:
    if col in loan_apps.columns:
        null_count = loan_apps[col].isna().sum()
        if null_count > 0:
            print(f"• loan_apps.{col}: {null_count:,} NULLs → Filling with 0")
            loan_apps[col] = loan_apps[col].fillna(0)

# loan_apps: collateral_type NULL when collateral_required = 0 → expected
if 'collateral_type' in loan_apps.columns:
    null_coll = loan_apps['collateral_type'].isna().sum()
    print(f"• loan_apps.collateral_type: {null_coll:,} NULLs → "
          f"Filling with 'None' for unsecured loans")
    loan_apps['collateral_type'] = loan_apps['collateral_type'].fillna('None')

# transactions: branch_id can have NaN for digital transactions
if 'branch_id' in transactions.columns:
    null_br = transactions['branch_id'].isna().sum()
    if null_br > 0:
        print(f"• transactions.branch_id: {null_br:,} NULLs → "
              f"Filling with -1 (online/digital)")
        transactions['branch_id'] = transactions['branch_id'].fillna(-1).astype(int)

# transactions: merchant_name can be NULL for non-merchant transactions
if 'merchant_name' in transactions.columns:
    null_merch = transactions['merchant_name'].isna().sum()
    if null_merch > 0:
        print(f"• transactions.merchant_name: {null_merch:,} NULLs → "
              f"Filling with 'Unknown'")
        transactions['merchant_name'] = transactions['merchant_name'].fillna('Unknown')

print("\n✅ Missing values handled!")

---
## 🔁 Step 4: Duplicate Detection

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 4: DUPLICATE DETECTION & REMOVAL
# ─────────────────────────────────────────────────────────────
# Check each table for duplicates in two ways:
#   1. Primary Key duplicates — same PK value appearing twice
#      (violates data integrity)
#   2. Full row duplicates — entire row is an exact copy
#
# If duplicates are found, we remove full-row duplicates using
# drop_duplicates() and re-assign the cleaned DataFrames.
#
# Primary keys checked:
#   regions→region_id, customers→customer_id,
#   loan_apps→application_id, transactions→transaction_id, etc.
# ─────────────────────────────────────────────────────────────

# Define primary keys for each table
primary_keys = {
    'regions':        'region_id',
    'loan_types':     'loan_type_id',
    'branches':       'branch_id',
    'employees':      'employee_id',
    'customers':      'customer_id',
    'credit_history': 'credit_history_id',
    'loan_apps':      'application_id',
    'loan_payments':  'payment_id',
    'transactions':   'transaction_id',
}

dup_rows = []
any_dups = False
for name, df in all_tables.items():
    pk = primary_keys.get(name, None)
    full_dups = df.duplicated().sum()
    pk_dups = df[pk].duplicated().sum() if pk and pk in df.columns else 'N/A'
    if (isinstance(pk_dups, int) and pk_dups > 0) or full_dups > 0:
        any_dups = True
    dup_rows.append({'Table': name, 'PK Column': pk or 'N/A',
                     'Total Rows': f"{len(df):,}",
                     'Duplicate PKs': pk_dups,
                     'Full Duplicates': full_dups})

display(pd.DataFrame(dup_rows))

if not any_dups:
    print("\n✅ No duplicate records found!")
else:
    print("\n⚠️  Duplicates detected — removing full duplicates...")
    for name in list(all_tables.keys()):
        df = all_tables[name]
        before = len(df)
        df = df.drop_duplicates()
        after = len(df)
        if before != after:
            print(f"  • {name}: Removed {before - after:,} duplicates")
            all_tables[name] = df
    # Re-assign cleaned tables
    regions = all_tables['regions']
    loan_types = all_tables['loan_types']
    branches = all_tables['branches']
    employees = all_tables['employees']
    customers = all_tables['customers']
    credit_history = all_tables['credit_history']
    loan_apps = all_tables['loan_apps']
    loan_payments = all_tables['loan_payments']
    transactions = all_tables['transactions']

---
## 📅 Step 5: Date Column Conversion

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 5: CONVERT DATE COLUMNS FROM STRING TO DATETIME
# ─────────────────────────────────────────────────────────────
# CSV files store dates as plain strings (e.g., '2023-04-15').
# We convert them to pandas datetime objects so we can:
#   • Extract year, month, quarter, day-of-week
#   • Calculate time differences (e.g., account tenure)
#   • Sort and filter by date ranges
#
# errors='coerce' converts unparseable dates to NaT (Not a Time)
# instead of raising an error. We track how many got coerced.
#
# Columns converted (11 total):
#   customers:     date_of_birth, account_open_date
#   employees:     date_of_birth, joining_date
#   branches:      established_date
#   credit_history: last_updated_date
#   loan_apps:     application_date, disbursement_date
#   loan_payments: due_date, payment_date
#   transactions:  transaction_date
# ─────────────────────────────────────────────────────────────

date_columns = {
    'customers':      ['date_of_birth', 'account_open_date'],
    'employees':      ['date_of_birth', 'joining_date'],
    'branches':       ['established_date'],
    'credit_history': ['last_updated_date'],
    'loan_apps':      ['application_date', 'disbursement_date'],
    'loan_payments':  ['due_date', 'payment_date'],
    'transactions':   ['transaction_date'],
}

conversion_rows = []
for tbl_name, cols in date_columns.items():
    df = all_tables[tbl_name]
    for col in cols:
        if col in df.columns:
            before_dtype = str(df[col].dtype)
            df[col] = pd.to_datetime(df[col], errors='coerce')
            invalid_dates = df[col].isna().sum()
            conversion_rows.append({
                'Table': tbl_name, 'Column': col,
                'Before': before_dtype, 'After': 'datetime64',
                'Invalid/Coerced': f"{invalid_dates:,}"
            })

display(pd.DataFrame(conversion_rows))
print("\n✅ All date columns converted!")

---
## 🏷️ Step 6: Categorical Column Standardization

### 6a. Unique Value Counts for Key Categorical Columns

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 6a: INSPECT CATEGORICAL COLUMNS
# ─────────────────────────────────────────────────────────────
# Before standardizing, we first inspect the unique values in
# each key categorical column to check for:
#   • Inconsistent casing (e.g., 'male' vs 'Male' vs 'MALE')
#   • Extra whitespace (e.g., ' Married ' vs 'Married')
#   • Unexpected categories or typos
#
# For each column, we display:
#   • Number of unique values
#   • Top 8 most frequent values with their counts
# ─────────────────────────────────────────────────────────────

categorical_checks = {
    'customers': ['gender', 'marital_status', 'education', 'employment_type',
                  'account_type', 'kyc_status', 'customer_segment', 'zone'],
    'employees': ['gender', 'designation', 'department'],
    'branches':  ['branch_type', 'zone'],
    'credit_history': ['credit_rating'],
    'loan_apps': ['status', 'loan_type_name'],
    'loan_payments': ['payment_status'],
    'transactions': ['transaction_type', 'transaction_mode', 'category', 'status'],
}

for tbl_name, cols in categorical_checks.items():
    df = all_tables[tbl_name]
    print(f"\n📌 {tbl_name.upper()}")
    for col in cols:
        if col in df.columns:
            uniques = df[col].nunique()
            vals = df[col].value_counts().head(8).to_dict()
            vals_str = ', '.join([f"{k}: {v:,}" for k, v in vals.items()])
            print(f"   {col} ({uniques} unique): {vals_str}")

### 6b. Strip Whitespace & Standardize Case

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 6b: CLEAN TEXT COLUMNS
# ─────────────────────────────────────────────────────────────
# Two cleaning operations:
#
# 1) STRIP WHITESPACE — Remove leading/trailing spaces from
#    ALL object (string) columns in ALL tables. This prevents
#    issues like ' Male' != 'Male' during groupby/filtering.
#
# 2) TITLE CASE — Standardize key categorical columns to Title
#    Case (e.g., 'MALE' → 'Male', 'married' → 'Married').
#    This ensures consistent values for grouping and charting.
#    Only applied to columns where Title Case makes sense
#    (not to free-text fields like names or descriptions).
# ─────────────────────────────────────────────────────────────

# Strip whitespace from all string/object columns
print("🔧 Stripping whitespace from all text columns...")
for name, df in all_tables.items():
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].str.strip()
print("✅ Whitespace stripped!")

# Standardize case for key columns
print("\n🔧 Standardizing case (Title Case) for key columns...")
title_case_cols = {
    'customers': ['gender', 'marital_status', 'education', 'employment_type',
                  'account_type', 'kyc_status', 'customer_segment'],
    'employees': ['gender', 'designation', 'department'],
    'branches':  ['branch_type'],
    'credit_history': ['credit_rating'],
    'loan_apps': ['status'],
    'loan_payments': ['payment_status'],
    'transactions': ['transaction_type', 'transaction_mode', 'status'],
}

for tbl_name, cols in title_case_cols.items():
    df = all_tables[tbl_name]
    for col in cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.title()

print("✅ Case standardization complete!")

---
## 📊 Step 7: Numerical Outlier Detection (IQR Method)

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 7: DETECT OUTLIERS USING IQR METHOD
# ─────────────────────────────────────────────────────────────
# The Interquartile Range (IQR) method flags values that fall
# outside [Q1 - 1.5×IQR, Q3 + 1.5×IQR] as potential outliers.
#
# For each numerical column, we display:
#   • Number and percentage of outliers detected
#   • Q1, Q3, and the computed lower/upper bounds
#
# IMPORTANT: Outliers are FLAGGED but NOT removed.
# In banking data, extreme values are often legitimate:
#   • High-value home loans (₹50L+)
#   • Premium customers with high income
#   • Large business transactions
# Removing them would distort the analysis.
# ─────────────────────────────────────────────────────────────

outlier_columns = {
    'customers':      ['annual_income', 'age'],
    'credit_history': ['credit_score', 'total_outstanding_debt',
                       'credit_utilization_pct', 'number_of_delinquencies'],
    'loan_apps':      ['loan_amount_requested', 'loan_amount_approved',
                       'tenure_months', 'interest_rate_pct'],
    'loan_payments':  ['emi_amount', 'penalty_amount', 'days_late',
                       'outstanding_balance'],
    'transactions':   ['amount'],
}


def detect_outliers_iqr(series, factor=1.5):
    """Detect outliers using IQR method and return count + bounds."""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR
    outliers = ((series < lower) | (series > upper)).sum()
    return outliers, lower, upper, Q1, Q3


outlier_rows = []
for tbl_name, cols in outlier_columns.items():
    df = all_tables[tbl_name]
    for col in cols:
        if col in df.columns:
            valid = df[col].dropna()
            if len(valid) == 0:
                continue
            outliers, lower, upper, q1, q3 = detect_outliers_iqr(valid)
            pct = (outliers / len(valid) * 100)
            outlier_rows.append({
                'Table': tbl_name, 'Column': col,
                'Outliers': f"{outliers:,}", '%': f"{pct:.2f}%",
                'Q1': f"{q1:,.2f}", 'Q3': f"{q3:,.2f}",
                'Lower Bound': f"{lower:,.2f}", 'Upper Bound': f"{upper:,.2f}"
            })

display(pd.DataFrame(outlier_rows))
print("\nℹ️  Note: Outliers are flagged for awareness — NOT removed.")
print("   Banking data often has legitimate extreme values (high-value loans, etc.)")

---
## ⚙️ Step 8: Feature Engineering — Derived Columns

### 8a. Customers — Age Group, Income Bracket, Account Tenure

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 8a: CUSTOMER FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────
# Create 3 new derived columns for the customers table:
#
# 1) age_group — Bin ages into 6 groups (18-25, 26-35, ...65+)
#    using pd.cut(). Useful for demographic segmentation and
#    comparing loan approval rates across age groups.
#
# 2) income_bracket — Categorize annual income into 5 tiers:
#    <2L, 2-5L, 5-10L, 10-20L, 20L+ (in Indian Rupees).
#    Helps analyze loan eligibility and spending patterns.
#
# 3) account_tenure_years — Calculate how many years each
#    customer has been with the bank (from account_open_date
#    to today). Indicates customer loyalty.
# ─────────────────────────────────────────────────────────────

# Age group
bins_age = [0, 25, 35, 45, 55, 65, 120]
labels_age = ['18-25', '26-35', '36-45', '46-55', '56-65', '65+']
customers['age_group'] = pd.cut(customers['age'], bins=bins_age, labels=labels_age)
print("✔ age_group:", customers['age_group'].value_counts().to_dict())

# Income bracket
bins_income = [0, 200000, 500000, 1000000, 2000000, float('inf')]
labels_income = ['<2L', '2-5L', '5-10L', '10-20L', '20L+']
customers['income_bracket'] = pd.cut(customers['annual_income'],
                                      bins=bins_income, labels=labels_income)
print("✔ income_bracket:", customers['income_bracket'].value_counts().to_dict())

# Account tenure in years
customers['account_tenure_years'] = (
    (pd.Timestamp.now() - customers['account_open_date']).dt.days / 365.25
).round(1)
print(f"✔ account_tenure_years — Mean: {customers['account_tenure_years'].mean():.1f} yrs")

display(customers[['customer_id', 'age', 'age_group', 'annual_income',
                    'income_bracket', 'account_tenure_years']].head())

### 8b. Credit History — Credit Band, High Utilization Flag

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 8b: CREDIT HISTORY FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────
# Create 2 new derived columns for the credit_history table:
#
# 1) credit_band — Bin credit scores into 4 risk categories:
#    Poor (<600), Fair (600-699), Good (700-799), Excellent (800+)
#    This is the standard credit score classification used by
#    banks for loan approval decisions.
#
# 2) high_utilization — Binary flag (0/1) indicating if the
#    customer's credit utilization exceeds 75%.
#    High utilization (>75%) is a red flag for credit risk —
#    it signals the customer is using most of their credit limit.
# ─────────────────────────────────────────────────────────────

# Credit score band
bins_credit = [0, 600, 700, 800, 900]
labels_credit = ['Poor (<600)', 'Fair (600-699)', 'Good (700-799)', 'Excellent (800+)']
credit_history['credit_band'] = pd.cut(credit_history['credit_score'],
                                        bins=bins_credit, labels=labels_credit)
print("✔ credit_band:", credit_history['credit_band'].value_counts().to_dict())

# High utilization flag
credit_history['high_utilization'] = (
    credit_history['credit_utilization_pct'] > 75
).astype(int)
print(f"✔ high_utilization: {credit_history['high_utilization'].sum():,} customers with >75% utilization")

display(credit_history[['customer_id', 'credit_score', 'credit_band',
                         'credit_utilization_pct', 'high_utilization']].head())

### 8c. Loan Applications — Approval Flag, Amount Gap, Date Parts

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 8c: LOAN APPLICATION FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────
# Create 6 new derived columns for the loan_apps table:
#
# 1) is_approved — Binary flag (0/1) for loans that were
#    approved (status = Disbursed, Approved, or Closed).
#    Makes it easy to calculate approval rates.
#
# 2) amount_gap — Difference between requested and approved
#    amount (₹). Positive = bank approved less than requested.
#
# 3) gap_pct — Gap as a percentage of requested amount.
#    Helps understand how much banks typically cut down.
#
# 4-6) app_year, app_month, app_quarter — Extracted from
#    application_date for time-series analysis and seasonality.
# ─────────────────────────────────────────────────────────────

# Approval flag
loan_apps['is_approved'] = loan_apps['status'].isin(
    ['Disbursed', 'Approved', 'Closed']
).astype(int)
print(f"✔ is_approved: {loan_apps['is_approved'].sum():,} approved out of {len(loan_apps):,}")

# Amount gap (requested - approved)
loan_apps['amount_gap'] = (
    loan_apps['loan_amount_requested'] - loan_apps['loan_amount_approved']
)
loan_apps['gap_pct'] = (
    loan_apps['amount_gap'] / loan_apps['loan_amount_requested'] * 100
).round(2)
print(f"✔ amount_gap & gap_pct — Mean gap: ₹{loan_apps['amount_gap'].mean():,.0f}")

# Application year & month
loan_apps['app_year'] = loan_apps['application_date'].dt.year
loan_apps['app_month'] = loan_apps['application_date'].dt.month
loan_apps['app_quarter'] = loan_apps['application_date'].dt.quarter
print("✔ app_year, app_month, app_quarter")

display(loan_apps[['application_id', 'status', 'is_approved',
                    'loan_amount_requested', 'loan_amount_approved',
                    'amount_gap', 'gap_pct']].head())

### 8d. Loan Payments — Late/Missed Flags, Date Parts

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 8d: LOAN PAYMENTS FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────
# Create 4 new derived columns for the loan_payments table:
#
# 1) is_late — Binary flag (0/1) if days_late > 0.
#    Captures both late AND missed payments.
#
# 2) is_missed — Binary flag (0/1) if payment_status='Missed'.
#    Specifically identifies completely missed EMIs (defaults).
#
# 3-4) pay_year, pay_month — Extracted from due_date for
#    tracking payment behavior trends over time.
#
# These flags are essential for default rate analysis and
# building risk models.
# ─────────────────────────────────────────────────────────────

# Late payment flag
loan_payments['is_late'] = (loan_payments['days_late'] > 0).astype(int)
print(f"✔ is_late: {loan_payments['is_late'].sum():,} late payments")

# Missed payment flag
loan_payments['is_missed'] = (
    loan_payments['payment_status'].str.lower() == 'missed'
).astype(int)
print(f"✔ is_missed: {loan_payments['is_missed'].sum():,} missed payments")

# Payment month
loan_payments['pay_year'] = loan_payments['due_date'].dt.year
loan_payments['pay_month'] = loan_payments['due_date'].dt.month
print("✔ pay_year, pay_month")

display(loan_payments[['payment_id', 'payment_status', 'days_late',
                        'is_late', 'is_missed', 'pay_year', 'pay_month']].head())

### 8e. Transactions — Date Parts, Amount Bucket

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 8e: TRANSACTIONS FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────
# Create 5 new derived columns for the transactions table:
#
# 1-4) txn_year, txn_month, txn_day_of_week, txn_hour —
#    Extracted from transaction_date for temporal analysis:
#    • Year/Month → yearly growth & seasonal trends
#    • Day of week → weekday vs weekend spending patterns
#    • Hour → peak transaction times (morning/evening/night)
#
# 5) amount_bucket — Categorize transaction amounts into 5 tiers:
#    Micro (<₹500), Small (₹500-2K), Medium (₹2K-10K),
#    Large (₹10K-50K), High Value (₹50K+).
#    Helps identify transaction size distribution.
# ─────────────────────────────────────────────────────────────

# Transaction year, month, day_of_week, hour
transactions['txn_year'] = transactions['transaction_date'].dt.year
transactions['txn_month'] = transactions['transaction_date'].dt.month
transactions['txn_day_of_week'] = transactions['transaction_date'].dt.day_name()
transactions['txn_hour'] = transactions['transaction_date'].dt.hour
print("✔ txn_year, txn_month, txn_day_of_week, txn_hour")

# Amount bucket
bins_txn = [0, 500, 2000, 10000, 50000, float('inf')]
labels_txn = ['Micro (<500)', 'Small (500-2K)', 'Medium (2K-10K)',
              'Large (10K-50K)', 'High Value (50K+)']
transactions['amount_bucket'] = pd.cut(transactions['amount'],
                                        bins=bins_txn, labels=labels_txn)
print("✔ amount_bucket:", transactions['amount_bucket'].value_counts().to_dict())

display(transactions[['transaction_id', 'amount', 'amount_bucket',
                       'txn_year', 'txn_month', 'txn_day_of_week']].head())

print("\n✅ Feature engineering complete!")

---
## 💾 Step 9: Final Summary & Cleaned Data Export

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 9a: FINAL SHAPE SUMMARY
# ─────────────────────────────────────────────────────────────
# Display the final shape of each table after all preprocessing
# steps, showing how many new columns were added via feature
# engineering compared to the original raw data.
#
# This gives a clear before-vs-after view of the preprocessing.
# ─────────────────────────────────────────────────────────────

# Update the all_tables dict with cleaned data
all_tables.update({
    'customers': customers,
    'credit_history': credit_history,
    'loan_apps': loan_apps,
    'loan_payments': loan_payments,
    'transactions': transactions,
})

original_cols = {
    'regions': 6, 'loan_types': 11, 'branches': 11, 'employees': 16,
    'customers': 29, 'credit_history': 12, 'loan_apps': 20,
    'loan_payments': 13, 'transactions': 13,
}

summary_rows = []
for name, df in all_tables.items():
    new_cols = df.shape[1] - original_cols.get(name, df.shape[1])
    new_str = f"+{new_cols}" if new_cols > 0 else "—"
    summary_rows.append({'Table': name, 'Rows': f"{df.shape[0]:,}",
                         'Columns': df.shape[1], 'New Columns': new_str})

display(pd.DataFrame(summary_rows))

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 9b: EXPORT CLEANED DATA TO CSV FILES
# ─────────────────────────────────────────────────────────────
# Save all 9 cleaned and feature-engineered DataFrames as CSV
# files in the 'eda/cleaned_data/' directory.
#
# Output files:
#   regions_cleaned.csv, loan_types_cleaned.csv,
#   branches_cleaned.csv, employees_cleaned.csv,
#   customers_cleaned.csv, credit_history_cleaned.csv,
#   loan_apps_cleaned.csv, loan_payments_cleaned.csv,
#   transactions_cleaned.csv
#
# These cleaned CSVs can be used directly by:
#   • eda_plotly.py / eda_plotly.ipynb for visualizations
#   • Power BI for dashboard creation
#   • Any ML pipeline for model training
# ─────────────────────────────────────────────────────────────

print(f"📁 Exporting cleaned data to: {CLEAN_PATH}\n")

export_rows = []
for name, df in all_tables.items():
    filepath = os.path.join(CLEAN_PATH, f"{name}_cleaned.csv")
    df.to_csv(filepath, index=False)
    size_mb = os.path.getsize(filepath) / (1024 ** 2)
    export_rows.append({'File': f"{name}_cleaned.csv", 'Size (MB)': f"{size_mb:.2f}"})
    print(f"  ✔ {name}_cleaned.csv ({size_mb:.2f} MB)")

display(pd.DataFrame(export_rows))

---
## ✅ Preprocessing Pipeline Complete!

### Summary of Steps Performed

| Step | Task | Status |
|------|------|--------|
| 1 | Data Loading & Shape Overview | ✔ Done |
| 2 | Data Types & Column Summary | ✔ Done |
| 3 | Missing Value Analysis & Handling | ✔ Done |
| 4 | Duplicate Detection & Removal | ✔ Done |
| 5 | Date Column Conversion (str → datetime) | ✔ Done |
| 6 | Categorical Standardization (strip + title case) | ✔ Done |
| 7 | Numerical Outlier Detection (IQR method) | ✔ Done |
| 8 | Feature Engineering (20+ derived columns) | ✔ Done |
| 9 | Cleaned Data Exported to CSV | ✔ Done |

### New Derived Columns Added

| Table | New Columns | Description |
|-------|------------|-------------|
| customers | `age_group`, `income_bracket`, `account_tenure_years` | Demographic segmentation |
| credit_history | `credit_band`, `high_utilization` | Risk categorization |
| loan_apps | `is_approved`, `amount_gap`, `gap_pct`, `app_year`, `app_month`, `app_quarter` | Approval analysis + time series |
| loan_payments | `is_late`, `is_missed`, `pay_year`, `pay_month` | Default tracking |
| transactions | `txn_year`, `txn_month`, `txn_day_of_week`, `txn_hour`, `amount_bucket` | Temporal + size analysis |

**Next Step →** Run `eda_plotly.ipynb` for visualizations!